# 🔍 Notebook 1 — Data Exploration
**Project:** Multilingual Fake News Detection (Uzbek, Russian, English)  
**Author:** Asliddin | Presidential School, Namangan  
**Series:** Asliddin Builds #02

---
In this notebook we:
1. Load and inspect all three language datasets
2. Analyze class distribution across languages
3. Explore text length distributions
4. Visualize word clouds per class
5. Prepare unified train/val/test split

In [ ]:
import sys
sys.path.append('../src')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')

## 1. Load Datasets

**Data sources:**
- English: LIAR dataset (Kaggle) + FakeNewsNet
- Russian: RuFake corpus (HuggingFace)
- Uzbek: Custom scraped dataset (kun.uz, daryo.uz, gazeta.uz)

All should be placed in `../data/raw/` as:
- `english.csv`
- `russian.csv`  
- `uzbek.csv`

Each CSV must have columns: `text`, `label` (real/fake/satire)

In [ ]:
raw_dir = '../data/raw'
dfs = {}

for lang, fname in [('en', 'english.csv'), ('ru', 'russian.csv'), ('uz', 'uzbek.csv')]:
    path = os.path.join(raw_dir, fname)
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['lang'] = lang
        dfs[lang] = df
        print(f'[{lang}] Loaded {len(df):,} samples')
    else:
        print(f'[{lang}] File not found: {path}')
        # Demo placeholder
        dfs[lang] = pd.DataFrame({
            'text':  [f'Sample {lang} text {i}' for i in range(100)],
            'label': np.random.choice(['real','fake','satire'], 100),
            'lang':  lang
        })
        print(f'  → Created demo placeholder with 100 samples')

# Combine
df_all = pd.concat(dfs.values(), ignore_index=True)
df_all['label'] = df_all['label'].str.lower().str.strip()
df_all = df_all[df_all['label'].isin(['real','fake','satire'])]
print(f'\nTotal combined: {len(df_all):,} samples')

## 2. Class × Language Distribution

In [ ]:
lang_names = {'en': 'English', 'ru': 'Russian', 'uz': 'Uzbek'}
label_colors = {'real': '#2d6a4f', 'fake': '#d62828', 'satire': '#f4a261'}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Class Distribution by Language', fontsize=14, fontweight='bold')

for ax, (lang, df_lang) in zip(axes, dfs.items()):
    counts = df_lang['label'].value_counts()
    colors = [label_colors.get(l, '#888') for l in counts.index]
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white')
    ax.set_title(f'{lang_names[lang]}\n(n={len(df_lang):,})', fontweight='bold')
    ax.set_ylabel('Count')
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, count + 5,
                str(count), ha='center', fontweight='bold', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n⚠️  Note: Uzbek dataset is smaller — this is expected for a low-resource language.')
print('We handle this with class-weighted loss and back-translation augmentation.')

## 3. Text Length Distribution

In [ ]:
df_all['text_len'] = df_all['text'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Text Length Distribution (words) by Language', fontsize=13, fontweight='bold')

colors_lang = {'en': '#4fc3f7', 'ru': '#ef5350', 'uz': '#66bb6a'}

for ax, (lang, df_lang) in zip(axes, dfs.items()):
    df_lang['text_len'] = df_lang['text'].apply(lambda x: len(str(x).split()))
    ax.hist(df_lang['text_len'].clip(0, 600), bins=40,
            color=colors_lang[lang], alpha=0.8, edgecolor='white')
    ax.axvline(256, color='red', linestyle='--', linewidth=1.5, label='Max tokens (256)')
    ax.set_title(lang_names[lang], fontweight='bold')
    ax.set_xlabel('Word count')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)
    median = df_lang['text_len'].median()
    ax.text(0.98, 0.95, f'Median: {median:.0f}w',
            transform=ax.transAxes, ha='right', va='top', fontsize=9)

plt.tight_layout()
plt.savefig('../results/text_length_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Train / Val / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Stratify by label AND language
df_all['strat_key'] = df_all['label'] + '_' + df_all['lang']

train_df, temp_df = train_test_split(df_all, test_size=0.30,
                                      stratify=df_all['strat_key'], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50,
                                      stratify=temp_df['strat_key'], random_state=42)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

os.makedirs('../data/processed', exist_ok=True)
for split, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    df[['text', 'label', 'lang']].to_csv(f'../data/processed/{split}.csv', index=False)

print('\nManifests saved to data/processed/')
print('Next: Notebook 02 — TF-IDF baseline')